# Chapter 9 worked exercise: the bike as the target

The chapter's pipeline predicting electric bikes from the trip and the pass type: baseline, logistic regression with odds ratios, confusion matrix, a tree and the depth sweep.

The chapter's pipeline, with the bike as the target: can you tell from a trip (and the pass type, which is now allowed as a feature) whether it was on an electric bike? Same steps, same scoring rule: every accuracy next to its baseline.

## 1. Load, clean, engineer

In [ ]:
# Excel: File > Open
import os
import pandas as pd

LOCAL = "../../data/raw/"
URL = "https://github.com/lshpaner/excel_to_python_exercises/releases/download/data/"
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
trips.shape

In [ ]:
# Excel: the keep column, the feature columns, and a helper for scoring
try:
    import model_metrics
except ImportError:                      # Colab does not have it yet
    %pip install -q model_metrics
from model_metrics import show_confusion_matrix, show_roc_curve, plot_threshold_metrics, summarize_model_performance

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

trips["start_time"] = pd.to_datetime(trips["start_time"])
trips = trips[(trips["duration"] > 1) & (trips["duration"] < 1440)].copy()   # clean rows

trips["log_duration"] = np.log(trips["duration"])
trips["hour"] = trips["start_time"].dt.hour
trips["weekend"] = (trips["start_time"].dt.weekday >= 5).astype(int)
trips["electric"] = (trips["bike_type"] == "electric").astype(int)
trips["round_trip"] = (trips["start_station"] == trips["end_station"]).astype(int)
trips["monthly"] = (trips["passholder_type"] == "Monthly Pass").astype(int)
trips["walkup"] = (trips["passholder_type"] == "Walk-up").astype(int)

BLUE, ORANGE, LIGHT = "#1F4E79", "#C55A11", "#9DB4CC"


def score(name, y_true, y_pred, baseline):
    """One line per model: accuracy against the baseline, precision and recall."""
    print(f"{name:<22} accuracy {accuracy_score(y_true, y_pred):.1%} (baseline {baseline:.1%})   "
          f"precision {precision_score(y_true, y_pred):.1%}   recall {recall_score(y_true, y_pred):.1%}")


len(trips)

## 2. Target, features, split

Target `electric`. Features: `log_duration`, `hour`, `weekend`, `round_trip`, `monthly`, `walkup`. An 80/20 stratified split with `random_state=42`. Print the baseline: the accuracy of always predicting the majority class.

In [ ]:
# Excel: cut the sheet into a training sheet and a test sheet
features = ["log_duration", "hour", "weekend", "round_trip", "monthly", "walkup"]
X, y = trips[features], trips["electric"]
# your code here: train_test_split, then the baseline accuracy of always predicting the majority class


## 3. Logistic regression

Fit, predict, `score`. Then the odds ratios. Which feature carries the most, and in which direction?

In [ ]:
# Excel: fit on the training sheet, score on the test sheet
# your code here: fit LogisticRegression, predict on X_test, call score(), then the odds ratios


*Your answer:* 

## 4. Confusion matrix

`show_confusion_matrix` with the model and the test rows. Which error is the model making more of, and why would a model do that on a 55-45 target?

In [ ]:
# Excel: the four cells behind the accuracy
# your code here: show_confusion_matrix on logit and the test rows


*Your answer 2:* 

## 5. A tree, the scorecard, and the depth sweep

A depth-3 tree on the same features (use `duration` in minutes in place of `log_duration` so the cutoffs read in minutes). Draw it. Put both models on one scorecard with `summarize_model_performance` (fit a second copy of the tree on the log features so both models can be scored on the same test frame). Then test accuracy for depths 1 to 20 and none: where does it peak?

In [ ]:
# Excel: the flowchart, then the memorization check
X_train_min = X_train.assign(duration=np.exp(X_train["log_duration"])).drop(columns="log_duration")
X_test_min = X_test.assign(duration=np.exp(X_test["log_duration"])).drop(columns="log_duration")
# your code here: depth-3 tree, score it, plot_tree, summarize_model_performance for both models, then the depth sweep


## 6. One sentence

The first question the tree asks is the pass type. Say in one sentence what that means for Metro's electric fleet, and name the thing the data cannot tell you (hint: chapter 8's solo exercise).

*Your sentence:* 

---

**Done early?** Drop `monthly` and `walkup` from the features and refit the logistic regression. How much of the accuracy was the pass type?